In [1]:
# ───────────────────  ROBUST INGESTION  WITH  TWO  PARTS  PLUS EXTRA HTML ───────────────────
import re, os, concurrent.futures as cf, errno
from pathlib import Path
import pandas as pd
from urllib.parse import urlparse

BASE_DIR     = Path("data")
CSV_FILE     = BASE_DIR / "index.csv"
PART_DIRS    = [BASE_DIR / "dataset-part-1"]
EXTRA_DIRS   = [
    ("html_validation/training/NotPhish", 0),
    ("html_validation/training/Phish",    1),
    ("html_validation/validation/NotPhish",0),
    ("html_validation/validation/Phish",   1),
]
MAX_BYTES    = 1_000_000_0     # read at most 1 MB/file
MIN_HTML_LEN = 20
THREADS      = min(8, os.cpu_count() or 1)

_rx_style = re.compile(rb"<style[\s\S]*?</style>", re.I)

def fast_read(path: Path, limit=MAX_BYTES, retry=2) -> str:
    while retry >= 0:
        try:
            with path.open("rb") as f:
                raw = f.read(limit)
            return _rx_style.sub(b"", raw).decode("utf-8", "ignore")
        except (TimeoutError, OSError) as e:
            if isinstance(e, OSError) and e.errno not in (errno.EAGAIN, errno.ETIMEDOUT):
                break
            retry -= 1
    return ""

def domain_from_website(val: str) -> str:
    if "://" in val:
        dom = urlparse(val).netloc.split(":")[0].lower()
        return dom[4:] if dom.startswith("www.") else dom
    return ""

def parse_row(rec):
    html_path, site = rec
    html = fast_read(html_path)
    return {
        "raw_html":   html,
        "domain":     domain_from_website(site),
        "num_forms":  html.count("<form"),
        "num_inputs": html.count("<input"),
        "num_iframes":html.count("<iframe"),
        "num_links":  html.count("<a "),
        "html_len":   len(html)
    }

# 1) Load and filter CSV
labels = (
    pd.read_csv(CSV_FILE, usecols=["website", "result"])
      .rename(columns={"result": "target"})
)

# 2) resolve html_path from PART_DIRS
def find_path(fname):
    for d in PART_DIRS:
        p = d / fname
        if p.exists():
            return p
    return None

labels["html_path"] = labels["website"].apply(find_path)
labels = labels[labels["html_path"].notnull()].reset_index(drop=True)
print(f"Using {len(labels)} pages found in parts 1 & 8")

# 3) parallel parse CSV-derived pages
with cf.ThreadPoolExecutor(max_workers=THREADS) as ex:
    rows = list(ex.map(
        parse_row,
        labels[["html_path","website"]]
          .itertuples(index=False, name=None),
        chunksize=128
    ))
df = pd.DataFrame(rows).join(labels["target"])

# 4) load extra HTML from local dirs
extra = []
for dir_path, lbl in EXTRA_DIRS:
    for path in Path(dir_path).rglob("*.html"):
        html = fast_read(path)
        if len(html) < MIN_HTML_LEN:
            continue
        extra.append({
            "raw_html": html,
            "domain":    "",      # no URL
            "num_forms": html.count("<form"),
            "num_inputs":html.count("<input"),
            "num_iframes":html.count("<iframe"),
            "num_links": html.count("<a "),
            "html_len": len(html),
            "target": lbl
        })
extra_df = pd.DataFrame(extra)
print(f"Loaded {len(extra_df)} extra pages from snapshots")

# 5) combine and clean
df = pd.concat([df, extra_df], ignore_index=True)
df = df[df["html_len"] >= MIN_HTML_LEN].reset_index(drop=True)
df.drop(columns="html_len", inplace=True)

# final dataset
data = df
print(f"Final dataset size: {len(data)} pages")
print("Label distribution:\n", data["target"].value_counts(normalize=True))
print("Average raw_html length:", data['raw_html'].str.len().mean())


Using 80000 pages found in parts 1 & 8
Loaded 12945 extra pages from snapshots
Final dataset size: 92723 pages
Label distribution:
 target
0    0.621971
1    0.378029
Name: proportion, dtype: float64
Average raw_html length: 110126.6943261111


In [3]:
# updated_html_feature_extract.py  – parity with extension JS extractor
from bs4 import BeautifulSoup
import re, math
from urllib.parse import urlparse

# ─── feature order (use exactly this for training) ───────────────────
NUM_ORDER = [
    # structural counts
    "forms","inputs","hidden_inputs","iframes","links","imgs","data_uri_imgs","scripts",
    # length & entropy
    "html_len","inline_js_len","entropy_html","entropy_js",
    # JavaScript red-flags
    "js_eval_cnt","js_suspicious_fn_cnt","num_event_handlers",
    # encoded / obfuscation
    "base64_cnt",
    # link-level flags & ratios
    "link_ip","link_at","punycode_link","js_href_link",
    "num_ext_links","ext_link_ratio",
    # form/action issues
    "form_empty_action","form_external_action","form_mailto",
    # iframe ratio etc.
    "inputs_per_form","iframe_ratio","js_html_ratio",
    # domain/TLD
    "domain_len","domain_hyphen","domain_digit","tld_suspicious","favicon_ext",
    # keyword stats
    "kw_cnt"
]

# ─── multilingual phishing vocab ─────────────────────────────────────
KW_EN = [
    "account","bank","confirm","password","passw0rd","p@ssword","p@55w0rd", "paxxword",
    "login","verify","credit card","ssn","social security","urgent",
    "immediately","click here","security","update","alert","expired","limited"
]
KW_PT = ["conta","banco","confirmar","senha","iniciar sessão","verificar",
         "urgente","clique aqui","atualize","segurança","aviso"]
KW_ES = ["cuenta","banco","confirmar","contraseña","iniciar sesión","verificar",
         "urgente","haz clic aquí","actualiza","seguridad","aviso"]
KW_RU = ["аккаунт","банк","подтвердите","пароль","срочно","нажмите здесь",
         "обновите","безопасность","ограничено"]
KW_ZH = ["账户","帐号","密码","登录","立即","点击","安全","验证","更新","警告"]
SUSPICIOUS_KW = KW_EN + KW_PT + KW_ES + KW_RU + KW_ZH

SUSPICIOUS_FUNCS = [
    "eval(", "atob(", "unescape(", "fromcharcode(", "document.write(",
    "settimeout(", "setinterval("
]

SUSPICIOUS_TLDS = {
    ".xyz",".top",".pw",".kim",".buzz",".click",".loan",".work",
    ".ru",".cn",".zip",".mov",".bond",".lol"
}

# ─── helpers ─────────────────────────────────────────────────────────
def shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    freq = {}
    for ch in text:
        freq[ch] = freq.get(ch, 0) + 1
    return -sum((c/len(text))*math.log2(c/len(text)) for c in freq.values())

# ─── main extractor ─────────────────────────────────────────────────
def extract_features(html: str, page_domain: str | None = None):
    soup = BeautifulSoup(html, "html.parser")
    feats = dict.fromkeys(NUM_ORDER, 0)

    # counts
    feats["forms"]   = len(soup.find_all("form"))
    feats["inputs"]  = len(soup.find_all("input"))
    feats["hidden_inputs"] = len(soup.find_all("input", {"type":"hidden"}))
    feats["iframes"] = len(soup.find_all("iframe"))
    feats["links"]   = len(soup.find_all("a"))
    feats["imgs"]    = len(soup.find_all("img"))
    feats["data_uri_imgs"] = len([img for img in soup.find_all("img", src=True)
                                  if img["src"].startswith("data:image")])
    feats["scripts"] = len(soup.find_all("script"))

    # raw strings
    html_lower = html.lower()
    inline_js  = " ".join(
        s.string or "" for s in soup.find_all("script") if not s.get("src")
    ).lower()

    # link flags
    num_ext = 0
    for a in soup.find_all("a", href=True):
        href = a["href"].lower()
        if page_domain and href.startswith("http") and page_domain not in href:
            num_ext += 1
        if re.search(r"http[s]?://\d{1,3}(?:\.\d{1,3}){3}", href):
            feats["link_ip"] = 1
        if "@" in href and not href.startswith("mailto:"):
            feats["link_at"] = 1
        if href.startswith("javascript:"):
            feats["js_href_link"] = 1
        if "xn--" in href:
            feats["punycode_link"] = 1
    feats["num_ext_links"] = num_ext
    feats["ext_link_ratio"] = num_ext / feats["links"] if feats["links"] else 0

    # forms
    for f in soup.find_all("form"):
        act = (f.get("action") or "").lower()
        if act in ("", "about:blank"):
            feats["form_empty_action"] = 1
        if act.startswith("mailto:"):
            feats["form_mailto"] = 1
        if page_domain and act.startswith("http") and page_domain not in urlparse(act).netloc:
            feats["form_external_action"] = 1

    # favicon origin
    icon = soup.find("link", rel=re.compile("icon", re.I))
    if icon and icon.get("href") and page_domain:
        href = icon["href"].lower()
        if href.startswith("http") and page_domain not in href:
            feats["favicon_ext"] = 1

    # JS red-flags
    feats["inline_js_len"]        = len(inline_js)
    feats["js_eval_cnt"]          = inline_js.count("eval(")
    feats["js_suspicious_fn_cnt"] = sum(inline_js.count(fn) for fn in SUSPICIOUS_FUNCS)
    feats["num_event_handlers"]   = len(re.findall(r"\son\w+\s*=", html_lower))
    feats["base64_cnt"]           = len(re.findall(r"base64[,;]", html_lower))

    # entropy / length
    feats["html_len"]     = len(html)
    feats["entropy_html"] = shannon_entropy(html_lower)
    feats["entropy_js"]   = shannon_entropy(inline_js)

    # ratios
    feats["inputs_per_form"] = feats["inputs"] / feats["forms"] if feats["forms"] else 0
    total_tags = len(soup.find_all(True)) or 1
    feats["iframe_ratio"]  = feats["iframes"] / total_tags
    feats["js_html_ratio"] = feats["inline_js_len"] / feats["html_len"] if feats["html_len"] else 0

    # domain-level
    if page_domain:
        dom = page_domain.lower()
        feats["domain_len"]    = len(dom)
        feats["domain_hyphen"] = int("-" in dom)
        feats["domain_digit"]  = int(any(c.isdigit() for c in dom))
        tld = "." + dom.split(".")[-1]
        feats["tld_suspicious"] = int(tld in SUSPICIOUS_TLDS)

    # keyword hits
    vis = soup.get_text(" ").lower()
    feats["kw_cnt"] = sum(int(kw in vis) for kw in SUSPICIOUS_KW)

    # return dict aligned with NUM_ORDER
    return {k: feats[k] for k in NUM_ORDER}, vis


In [4]:
# # ─────────────────────────── Pipeline: feature → XGBoost ──────────────────────────
# print("Stage 1 ▸ sampling 30 000 pages …")
# subset = data.sample(n=30_000, random_state=42).reset_index(drop=True)

# # ------------------------------------------------------------------- feature extract
# print("Stage 2 ▸ extracting features with extract_features() …")

# feat_rows = []
# for idx, row in subset.iterrows():
#     feats, _ = extract_features(row["raw_html"], row["domain"])
#     feat_rows.append(feats)
#     if (idx + 1) % 5000 == 0:
#         print(f"  • {idx+1} / 30000 done")

# X = (
#     pd.DataFrame(feat_rows)[NUM_ORDER]  # keep training order
#       .astype("float32")
#       .fillna(0)
# )
# y = subset["target"].astype("int8")
# print("Stage 2 ✔︎  Feature matrix shape:", X.shape)

# # ------------------------------------------------------------------- split
# print("Stage 3 ▸ splitting 20 000 train / 10 000 valid …")
# from sklearn.model_selection import train_test_split
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y,
#     train_size=20_000, test_size=10_000,
#     random_state=42, stratify=y
# )

# # ------------------------------------------------------------------- grid-search
# print("Stage 4 ▸ grid-searching XGBoost hyper-params …")
# from xgboost import XGBClassifier
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import roc_auc_score, classification_report

# param_grid = {
#     "n_estimators":  [200, 400, 600],
#     "max_depth":     [4, 6, 8],
#     "learning_rate": [0.05, 0.1, 0.2],
# }
# base_model = XGBClassifier(
#     objective="binary:logistic",
#     eval_metric="auc",
#     tree_method="hist",
#     n_jobs=-1,
#     random_state=42,
# )

# grid = GridSearchCV(
#     base_model,
#     param_grid,
#     cv=3,
#     scoring="roc_auc",
#     n_jobs=-1,
#     verbose=1,
# )
# grid.fit(X_train, y_train)

# print("\nStage 4 ✔︎  Best params:", grid.best_params_)
# print("Best CV AUC:", grid.best_score_)

# # ------------------------------------------------------------------- validation metrics
# best_model = grid.best_estimator_
# train_auc = roc_auc_score(y_train, best_model.predict_proba(X_train)[:, 1])
# val_auc   = roc_auc_score(y_val,   best_model.predict_proba(X_val)[:, 1])
# print(f"\nStage 5 ▸ train AUC={train_auc:.4f} • val AUC={val_auc:.4f}\n")
# print(classification_report(y_val, best_model.predict(X_val)))

# # ------------------------------------------------------------------- final fit & save
# print("Stage 6 ▸ retraining on full 20 000-sample train set …")
# best_model.fit(X_train, y_train)
# best_model.save_model("xgb_final_model.json")
# print("Model saved to xgb_final_model.json ✔︎")


Stage 1 ▸ sampling 30 000 pages …
Stage 2 ▸ extracting features with extract_features() …
  • 5000 / 30000 done
  • 10000 / 30000 done
  • 15000 / 30000 done
  • 20000 / 30000 done
  • 25000 / 30000 done
  • 30000 / 30000 done
Stage 2 ✔︎  Feature matrix shape: (30000, 34)
Stage 3 ▸ splitting 20 000 train / 10 000 valid …
Stage 4 ▸ grid-searching XGBoost hyper-params …
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Stage 4 ✔︎  Best params: {'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 600}
Best CV AUC: 0.9785585286215124

Stage 5 ▸ train AUC=0.9997 • val AUC=0.9814

              precision    recall  f1-score   support

           0       0.95      0.96      0.95      6253
           1       0.93      0.91      0.92      3747

    accuracy                           0.94     10000
   macro avg       0.94      0.93      0.94     10000
weighted avg       0.94      0.94      0.94     10000

Stage 6 ▸ retraining on full 20 000-sample train set …
Model saved to xgb_fi

In [4]:
# ─────────────────────────── Pipeline: feature → XGBoost (use all data, 70/15/15) ──────────────────────────
print("Stage 1 ▸ extracting features on all pages …")
feat_rows = []
for idx, row in data.iterrows():
    feats, _ = extract_features(row["raw_html"], row["domain"])
    feat_rows.append(feats)
    if (idx + 1) % 5000 == 0:
        print(f"  • {idx+1} / {len(data)} done")
X = pd.DataFrame(feat_rows)[NUM_ORDER].astype("float32").fillna(0)
y = data["target"].astype("int8")
print("Stage 1 ✔︎  Feature matrix shape:", X.shape)

# ------------------------------------------------------------------- split 70/15/15
print("Stage 2 ▸ splitting 70% train / 15% val / 15% test …")
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, train_size=0.7, test_size=0.3,
    random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, train_size=0.5, test_size=0.5,
    random_state=42, stratify=y_temp
)
print(f"  • train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

# ------------------------------------------------------------------- tune on train
print("Stage 3 ▸ tuning hyperparameters …")
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report

param_grid = {
    "n_estimators":  [200, 400, 600],
    "max_depth":     [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
}
base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)
grid = GridSearchCV(base, param_grid, cv=3, scoring="roc_auc", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
print("Stage 3 ✔︎  Best params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)

# ------------------------------------------------------------------- validate
best_model = grid.best_estimator_
train_auc = roc_auc_score(y_train, best_model.predict_proba(X_train)[:,1])
val_auc   = roc_auc_score(y_val,   best_model.predict_proba(X_val)[:,1])
print(f"Stage 4 ▸ train AUC={train_auc:.4f} • val AUC={val_auc:.4f}")
print(classification_report(y_val, best_model.predict(X_val)))

# ------------------------------------------------------------------- retrain & save
print("Stage 5 ▸ retraining on train+val …")
X_comb = pd.concat([X_train, X_val])
y_comb = pd.concat([y_train, y_val])
best_model.fit(X_comb, y_comb)
best_model.save_model("xgb_final_model.json")
print("Model saved to xgb_final__final_model.json")

# ------------------------------------------------------------------- test
print("Stage 6 ▸ final test evaluation …")
test_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1])
print(f"Test AUC={test_auc:.4f}")
print(classification_report(y_test, best_model.predict(X_test)))


Stage 1 ▸ extracting features on all pages …
  • 5000 / 92723 done
  • 10000 / 92723 done
  • 15000 / 92723 done
  • 20000 / 92723 done
  • 25000 / 92723 done
  • 30000 / 92723 done
  • 35000 / 92723 done
  • 40000 / 92723 done
  • 45000 / 92723 done
  • 50000 / 92723 done
  • 55000 / 92723 done
  • 60000 / 92723 done
  • 65000 / 92723 done
  • 70000 / 92723 done
  • 75000 / 92723 done
  • 80000 / 92723 done
  • 85000 / 92723 done
  • 90000 / 92723 done
Stage 1 ✔︎  Feature matrix shape: (92723, 34)
Stage 2 ▸ splitting 70% train / 15% val / 15% test …
  • train: 64906, val: 13908, test: 13909
Stage 3 ▸ tuning hyperparameters …
Fitting 3 folds for each of 27 candidates, totalling 81 fits
Stage 3 ✔︎  Best params: {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 600}
Best CV AUC: 0.9841333830886169
Stage 4 ▸ train AUC=0.9999 • val AUC=0.9881
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      8650
           1       0.95      0.94  

In [6]:
# ────────── Test & report mispredicted HTML lengths ──────────
import pandas as pd
from pathlib import Path
from xgboost import XGBClassifier

# 1) load trained model
model = XGBClassifier()
model.load_model("xgb_final_model.json")

# 2) gather .html files
html_dir = Path("snapshots")
files = list(html_dir.rglob("*.html"))
print(f"Found {len(files)} HTML files in '{html_dir}'")

# 3) extract features
feat_rows = []
for i, path in enumerate(files, 1):
    html = path.read_text(encoding="utf-8", errors="ignore")
    feats, _ = extract_features(html, None)
    feat_rows.append(feats)
    if i % 100 == 0:
        print(f"  • {i}/{len(files)} processed")
X_test = pd.DataFrame(feat_rows)[NUM_ORDER].astype("float32").fillna(0)

# 4) predict
probs = model.predict_proba(X_test)[:, 1]
preds = model.predict(X_test)

# 5) show sample & detection rate
results = pd.DataFrame({"file": [p.name for p in files], "phish_prob": probs, "pred": preds})
print("\nSample predictions:")
print(results.head(10))
detected = (probs > 0.5).sum()
print(f"\nDetected phishing: {detected}/{len(files)} ({detected/len(files):.1%})")

# 6) lengths of mispredicted (false negatives)
wrong_lengths = []
for path, pred in zip(files, preds):
    if pred == 1:  # missed phishing
        wrong_lengths.append(len(path.read_text(encoding="utf-8", errors="ignore")))
print("\nLengths of mispredicted HTML files:", wrong_lengths)


Found 413 HTML files in 'snapshots'
  • 100/413 processed
  • 200/413 processed
  • 300/413 processed
  • 400/413 processed

Sample predictions:
                                                file  phish_prob  pred
0  310_lite.evernote.com_note_def7c487-d207-e873-...    0.995134     1
1   2280_klockners.es_cgi-sys_suspendedpage.cgi.html    0.799717     1
2                   2453_reagan2021.weebly.com_.html    0.995191     1
3                     2358_livraison-track.com_.html    0.213951     0
4             212_supporttelkommail.weebly.com_.html    0.996659     1
5    2851_sordid-odd-buffet.glitch.me_care.html.html    0.746573     1
6                             1370_ln.run_m2tMt.html    0.292359     0
7              4697_kucoienlogiines.webflow.io_.html    0.345577     0
8    4023_visa.co.jp.bornaland.com_visa-secure_.html    0.599499     1
9                            45_www.google.com_.html    0.735456     1

Detected phishing: 215/413 (52.1%)

Lengths of mispredicted HTML files: [